# Prerequisites
1. Need to have IBM ILOG CPLEX Optimization Studio installed locally on your machine.
2. Need to have a python environment, we recommend Conda: Install Conda from https://www.anaconda.com/download/success?reg=skipped 

For students: https://academic.ibm.com/a2mt/
1. Sign in with an IBM account
2. Click on Data Science 
3. Click on ILOG CPLEX Optimization Studio
4. Select HTTP as Download method and choose the OS download option for your system

Otherwise:
https://www.ibm.com/products/ilog-cplex-optimization-studio/pricing

To create the environment:

In [ ]:
conda env create --name <name> --file environment.yml

To activate the environment:

In [ ]:
conda activate <name>

If you get an error make sure all libraries are installed inside of your environment.

Specifically check:
1. docplex
2. networkx
3. natsort
4. cplex

To install a library in the environment make sure the environment is active then:

In [ ]:
pip install <name>

These are the imports used to run CPLEX as well as read the graphs.

In [ ]:
# Imports
import os
import json
import cplex
import docplex
import numpy as np
import networkx as nx
from math import sqrt
from natsort import natsorted
from docplex.mp.model import Model

read_nxgraph reads the input file and constructs the graph based on the contents of the file.

In [7]:
def read_nxgraph(filename: str) -> nx.Graph(): # type: ignore
    graph = nx.Graph()
    with open(filename, 'r') as file:
        line = file.readline()
        is_first_line = True
        while line is not None and line != '':
            if '//' not in line:
                if is_first_line:
                    strings = line.split(" ")
                    num_nodes = int(strings[0])
                    num_edges = int(strings[1])
                    nodes = list(range(num_nodes))
                    graph.add_nodes_from(nodes)
                    is_first_line = False
                else:
                    node1, node2, weight = line.split()
                    graph.add_edge(int(node1), int(node2), weight=weight)
            line = file.readline()
    return graph

cplex_maxcut constructs the COPT solver and utilizes it to solve a given graph.

In [ ]:
def cplex_maxcut(graph):

    # Create QUBO matrix
    nodes = len(list(graph.nodes))
    J = nx.to_numpy_array(graph)
    
    # Construct model
    model = Model("maxcut_qubo")

    # Set time limit 1 hr
    model.set_time_limit(900)
    model.set_log_output(True) 
    model.parameters.mip.tolerances.mipgap=0.0

    # Create variable for each vertex
    x = model.binary_var_list(nodes)

    # Objective function
    objective = model.sum(
        -J[i, j] * (2 * x[i] - 1) * (2 * x[j] - 1) * (1/sqrt(nodes))
        for i in range(nodes) for j in range(i + 1, nodes) if J[i, j] != 0.0
    )
    model.minimize(objective)
    
    # Solve
    solution = model.solve()
    sol = ""
    for i in range(nodes):
        sol += f"{int(x[i].solution_value)}"
    return float(solution.objective_value), sol
    

Test code to run the solver.

In [ ]:
# Parse all files in data_dir 
data_dir = "../tutorials/VCA/dataset/EA/EA_10x10" #Change between 10x10, 20x20, 40x40
input_files = natsorted(os.listdir(data_dir))

# Write directories & files
results_dir = f"../results/{'/'.join(data_dir.split('/')[2:])}"
solutions_dir = f"../solutions/{'/'.join(data_dir.split('/')[2:])}"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(solutions_dir, exist_ok=True)

i = 0
for file in input_files:
    i += 1
    if i > 25: break
    
    graph = read_nxgraph(f'{data_dir}/{file}') 
    obj_val, sol = cplex_maxcut(graph)

    with open(f"{results_dir}/CPLEX.txt", "a") as f:
        f.write(f"{obj_val}\n")

    with open(f"{solutions_dir}/CPLEX.txt", "a") as f:
        f.write(f"{sol}\n")
